In [1]:
 !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from langchain_core.documents import Document

In [3]:
sample_doc = Document(
    page_content="Hello World!",
    metadata={"source": "https://www.google.com"}
)

In [4]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [5]:
# Text data
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data/Python.txt", encoding="utf-8")

/var/folders/5n/cjkg0wbd1cj5zjsb_k5g3_cr0000gn/T/ipykernel_1910/2888318756.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [6]:
document = loader.load()

In [7]:
document

[Document(metadata={'source': 'data/Python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

In [8]:
 # PDF data
 from langchain_community.document_loaders.pdf import PyPDFLoader

 pdf_loader = PyPDFLoader("data/IEEE_Conference_Template__2_.pdf")

 document = pdf_loader.load()
 document

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-07-16T04:20:33+00:00', 'author': '', 'keywords': '', 'moddate': '2026-07-16T04:20:33+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/IEEE_Conference_Template__2_.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content='Detection of Malware in Android System.\nHarini (24BCS009) Mokshith (24BCS106)\nDept. of Computer Science and Engineering Dept. of Computer Science and Engineering\nIIITDM Jabalpur IIITDM Jabalpur\nJabalpur, India Jabalpur, India\nAkhil Kumar Valluru (24BCS018) Chatur (24BCS021)\nDept. of Computer Science and Engineering Dept. of Computer Science and Engineering\nIIITDM Jabalpur IIITDM Jabalpur\nJabalpur, India Jabalpur, India\nRanjith Reddy (24BCS057)\nDept. of Computer Science and Engineering\nIIITDM Jabalpur\nJabalpur, 

## ingestion pipeline

In [9]:
# Data => Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [10]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [11]:
all_pdf_documents = load_all_pdfs()

total pdfs: 3
total pages: 161


In [12]:
# chunks
!pip install langchain_text_splitters


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [14]:
chunks = split_docs(all_pdf_documents)

In [15]:
len(chunks)

170

 ## embedding

In [16]:
from sentence_transformers import SentenceTransformer


W1024 13:27:34.736000 1910 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [17]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        
        self.model_name=model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [18]:
embedding_manager = EmbeddingManager()


loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


/var/folders/5n/cjkg0wbd1cj5zjsb_k5g3_cr0000gn/T/ipykernel_1910/4021223045.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


## vector stroe


In [19]:
import chromadb
import uuid

In [20]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

In [21]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 170


In [22]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

embeddings shape: (170, 384)
total documents added in vector store= 170
docs in collection: 340


 ## retrieval pipeline

In [23]:
from sklearn.metrics.pairwise import cosine_similarity

In [24]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [25]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [26]:
rag_retriever.retrieve("What is Linear")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 4 documents


[{'id': 'doc_33cc58c9-3419-4474-8448-6de280023793',
  'document': 'Linear Regression\n• Linear Regression is a supervised machine learning algorithm.\n• It tries to find out the best linear relationship that describes the \ndata you have i.e. .\n          Given: (X, Y) training examples\n          Goal: find a (hypothesis) function  (h) which predicts y for \n                     given x\n• In regression y is continuous value i.e., real numbers.\n• Input feature X can be comprise of a  single feature or multiple  \nfeature',
  'metadata': {'author': 'OpenClassroom',
   'source': 'data/pdfs/Lectures 4,5 and 6.pdf',
   'producer': 'Microsoft® PowerPoint® for Microsoft 365',
   'page_label': '2',
   'moddate': '2026-08-19T18:13:13+05:30',
   'page': 1,
   'creator': 'Microsoft® PowerPoint® for Microsoft 365',
   'creationdate': '2026-08-19T18:13:13+05:30',
   'total_pages': 66,
   'content_length': 452,
   'doc_index': 1,
   'title': 'Types of Learning'},
  'distance': 0.9195634126663208,

## integrating with LLM

In [34]:
API_Key_GROQ="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"

In [42]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=API_Key_GROQ,
    model="openai/gpt-oss-120b",
    temperature=0.1,
    max_tokens=1024
)

In [43]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke([prompt.format(context=context, query=query)]) # expecting a list as prompt
    return response.content

In [44]:
answer = generate_output("what is Linear regression?", rag_retriever, llm)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 3 documents


In [45]:
print(answer)

**Linear regression** is a **supervised machine‑learning algorithm** used for **regression tasks**, where the target variable \(y\) is a continuous (real‑valued) quantity.  

Given a set of training examples \((X, Y)\) – where each \(X\) may be a single feature or a vector of multiple features – linear regression seeks a **hypothesis function** \(h\) (typically written as \(h_\theta(x) = \theta_0 + \theta_1 x_1 + \dots + \theta_n x_n\)) that best predicts the output \(y\) for any input \(x\).

The algorithm works by finding the **best‑fit line** (in one dimension) or **best‑fit hyperplane** (in higher dimensions) that minimizes the overall prediction error (commonly the sum of squared residuals). In other words, it fits a straight line through the data points such that the total distance between the observed values and the line is as small as possible. This line then serves as the model for making future predictions.
